# LeetCode 2109 — Adding Spaces to a String bằng Python

Link đề: https://leetcode.com/problems/adding-spaces-to-a-string/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, String, Simulation

Skeleton LeetCode:

```python
class Solution:
    def addSpaces(self, s: str, spaces: List[int]) -> str:
        ...
```

Lộ trình:
1. **Chèn trực tiếp** từng dấu cách vào chuỗi (từ **cuối về đầu** để chỉ số không bị lệch) — `O(n · m)`.
2. **Hai con trỏ** duyệt `s` và `spaces` song song, ghi vào list ký tự rồi `join` — `O(n + m)` (**nộp bài**).
3. **Cắt lát giữa các vị trí chèn** rồi `" ".join(...)` — `O(n + m)`, ít vòng lặp Python nhất.
4. **Tập vị trí + duyệt ký tự** — `O(n + m)`, không cần `spaces` tăng dần.

## 1. Mục tiêu học

1. Hiểu vì sao **chuỗi Python bất biến** khiến chèn lặp đi lặp lại tốn `O(n)` mỗi lần, và cách tránh bằng list + `join`.
2. Dùng **hai con trỏ** trên hai dãy tăng dần (vị trí ký tự và vị trí chèn).
3. Nhận ra lỗi **lệch chỉ số** khi chèn từ trái sang phải trên chuỗi đang thay đổi, và hai cách sửa (chèn từ phải, hoặc cộng offset).
4. Viết lời giải một dòng bằng cắt lát + `join`, và lời giải dùng `set` khi `spaces` không sắp xếp.
5. Đo hiệu năng để thấy khác biệt `O(n·m)` và `O(n + m)` với `n, m = 3·10⁵`.

## 2. Đề bài & ví dụ

Cho chuỗi `s` và mảng tăng ngặt `spaces` gồm các chỉ số trong `s`. Chèn một dấu cách **trước** ký tự ở mỗi chỉ số đó (chỉ số tính trên chuỗi **gốc**). Trả về chuỗi kết quả.

**Ví dụ 1:** `s = "LeetcodeHelpsMeLearn"`, `spaces = [8,13,15]` → `"Leetcode Helps Me Learn"`.

**Ví dụ 2:** `s = "icodeinpython"`, `spaces = [1,5,7,9]` → `"i code in py thon"`.

**Ví dụ 3:** `s = "spacing"`, `spaces = [0,1,2,3,4,5,6]` → `" s p a c i n g"` (được chèn trước ký tự đầu).

**Ràng buộc:** `1 ≤ len(s) ≤ 3·10⁵`, `s` gồm chữ cái tiếng Anh; `1 ≤ len(spaces) ≤ 3·10⁵`, `0 ≤ spaces[i] ≤ len(s) − 1`, tăng ngặt.

In [1]:
import random
import string
import time
from typing import List

examples = [(("LeetcodeHelpsMeLearn", [8, 13, 15]), "Leetcode Helps Me Learn"),
            (("icodeinpython", [1, 5, 7, 9]), "i code in py thon"),
            (("spacing", [0, 1, 2, 3, 4, 5, 6]), " s p a c i n g")]
for (s, sp), want in examples:
    print(f"{s!r}, {sp} -> {want!r}")

'LeetcodeHelpsMeLearn', [8, 13, 15] -> 'Leetcode Helps Me Learn'
'icodeinpython', [1, 5, 7, 9] -> 'i code in py thon'
'spacing', [0, 1, 2, 3, 4, 5, 6] -> ' s p a c i n g'


## 3. Kiến thức nền

### 3.1 Chuỗi bất biến và `join`

`s[:p] + " " + s[p:]` tạo **chuỗi mới** dài `n + 1` — tốn `O(n)`. Ghép nhiều mảnh nên gom vào list rồi `"".join(list)` một lần (`O(tổng độ dài)`).

In [2]:
parts = ["Leet", "code", " ", "Helps"]
print("".join(parts), "|", " ".join(["a", "b", "c"]), "|", repr(" ".join(["", "s", "p"])))

Leetcode Helps | a b c | ' s p'


### 3.2 Hai con trỏ trên hai dãy tăng dần

Con trỏ `i` chạy trên `s`, con trỏ `j` chạy trên `spaces`. Khi `i == spaces[j]` thì chèn dấu cách và tăng `j`. Vì cả hai dãy tăng, mỗi con trỏ chỉ đi tới.

### 3.3 Cắt lát theo các mốc

Các mốc `0 = b₀ ≤ b₁ < … < b_m ≤ n` chia `s` thành `m + 1` mảnh `s[b_t : b_{t+1}]`. Nối các mảnh bằng `" "` chính là chèn dấu cách ở các mốc `b₁..b_m`.

In [3]:
s, sp = "icodeinpython", [1, 5, 7, 9]
bounds = [0] + sp + [len(s)]
pieces = [s[a:b] for a, b in zip(bounds, bounds[1:])]
print(pieces, "->", " ".join(pieces))

['i', 'code', 'in', 'py', 'thon'] -> i code in py thon


### 3.4 Chuỗi bất biến: mỗi phép "sửa" là tạo đối tượng mới

`id()` cho thấy sau `t = t + "x"` biến `t` trỏ tới một đối tượng **khác** (trừ khi CPython tối ưu nối tại chỗ). List thì sửa tại chỗ: `append` không đổi `id`.

In [4]:
t = "abc"; a = id(t); u = t; t = t + "x"
print("chuỗi: cùng đối tượng sau khi nối?", id(t) == a, "| bản cũ vẫn còn:", u)
L = list("abc"); b = id(L); L.append("x")
print("list: cùng đối tượng sau append?", id(L) == b, "|", L)

chuỗi: cùng đối tượng sau khi nối? False | bản cũ vẫn còn: abc
list: cùng đối tượng sau append? True | ['a', 'b', 'c', 'x']


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2 — đặt chỉ số dưới từng ký tự, đánh dấu `^` ở chỉ số cần chèn:

```
chỉ số:  0 1 2 3 4 5 6 7 8 9 10 11 12
s:       i c o d e i n p y t h  o  n
chèn:      ^       ^   ^   ^
kết quả: i _code _in _py _thon   ->  "i code in py thon"
```

Kết quả dài đúng `n + m = 13 + 4 = 17`. Mỗi ký tự gốc xuất hiện đúng một lần và theo thứ tự cũ; trước ký tự ở chỉ số `spaces[j]` có đúng một dấu cách.

**Bẫy:** nếu chèn dấu cách ở chỉ số `1` trước, chuỗi dài thêm 1 ⇒ ký tự gốc ở chỉ số `5` giờ ở chỉ số `6`. Chèn tiếp ở "chỉ số 5" sẽ sai vị trí.

In [5]:
s, sp = "icodeinpython", [1, 5, 7, 9]
t = s
for p in sp:
    t = t[:p] + " " + t[p:]                    # chèn theo chỉ số GỐC trên chuỗi ĐÃ đổi
print("chèn trái -> phải không bù:", repr(t))
t = s
for p in reversed(sp):
    t = t[:p] + " " + t[p:]                    # chèn từ phải: phần bên trái chưa bị dịch
print("chèn phải -> trái:         ", repr(t))

chèn trái -> phải không bù: 'i cod e i npython'
chèn phải -> trái:          'i code in py thon'


## 5. Cách 1 — Brute Force / baseline: chèn từ phải sang trái

**Ý tưởng:** chèn ở vị trí lớn nhất trước. Các chỉ số nhỏ hơn nằm bên trái chỗ chèn nên **không bị dịch**.

In [6]:
def add_spaces_insert(s, spaces):
    for p in reversed(spaces):
        s = s[:p] + " " + s[p:]
    return s

for (s, sp), want in examples:
    print(repr(add_spaces_insert(s, sp)), "kỳ vọng", repr(want))

'Leetcode Helps Me Learn' kỳ vọng 'Leetcode Helps Me Learn'
'i code in py thon' kỳ vọng 'i code in py thon'
' s p a c i n g' kỳ vọng ' s p a c i n g'


**Độ phức tạp:** mỗi lần chèn tạo chuỗi mới `O(n + m)` ⇒ `O(m · (n + m))` — tới `9·10¹⁰` ký tự sao chép với `n = m = 3·10⁵`. Space `O(n + m)`.

**Nút thắt:** sao chép toàn bộ chuỗi ở **mỗi** lần chèn.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** kết quả có cấu trúc đơn giản: đi qua `s` từ trái sang phải, **trước** khi ghi ký tự `s[i]`, nếu `i` là vị trí chèn kế tiếp thì ghi `" "`. Ghi vào list, cuối cùng `join` một lần ⇒ mỗi ký tự được chép đúng một lần.

**Invariant hai con trỏ:** khi xét `s[i]`, `j` là chỉ số nhỏ nhất với `spaces[j] ≥ i` (các vị trí chèn `< i` đã xử lý).

**Quy tắc:** *nếu* `j < m` và `spaces[j] == i` *thì* ghi `" "` và `j += 1`; luôn ghi `s[i]`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai con trỏ + list (code nộp)

In [7]:
def add_spaces_two_pointers(s, spaces):
    out = []
    j, m = 0, len(spaces)
    for i, ch in enumerate(s):
        if j < m and spaces[j] == i:
            out.append(" ")
            j += 1
        out.append(ch)
    return "".join(out)

for (s, sp), want in examples:
    print(repr(add_spaces_two_pointers(s, sp)), "kỳ vọng", repr(want))

'Leetcode Helps Me Learn' kỳ vọng 'Leetcode Helps Me Learn'
'i code in py thon' kỳ vọng 'i code in py thon'
' s p a c i n g' kỳ vọng ' s p a c i n g'


Code nộp LeetCode của Cách 2: xem mục 9.

**Độ phức tạp:** `O(n + m)` thời gian, `O(n + m)` bộ nhớ cho list kết quả.

### 7.2 Cách 3 — Cắt lát + `" ".join`

In [8]:
def add_spaces_slices(s, spaces):
    bounds = [0] + spaces + [len(s)]
    return " ".join(s[a:b] for a, b in zip(bounds, bounds[1:]))

for (s, sp), want in examples:
    print(repr(add_spaces_slices(s, sp)), "kỳ vọng", repr(want))

'Leetcode Helps Me Learn' kỳ vọng 'Leetcode Helps Me Learn'
'i code in py thon' kỳ vọng 'i code in py thon'
' s p a c i n g' kỳ vọng ' s p a c i n g'


Khi `spaces[0] = 0`, mảnh đầu là `s[0:0] = ""` ⇒ `join` tạo dấu cách ở đầu — đúng ví dụ 3. Vì `spaces` tăng **ngặt**, không có mảnh rỗng ở giữa.

**Độ phức tạp:** `O(n + m)`; chỉ có `m + 1` vòng lặp (lát cắt chạy trong C) ⇒ nhanh nhất trong Python.

### 7.3 Cách 4 — Tập vị trí + duyệt ký tự

In [9]:
def add_spaces_set(s, spaces):
    pos = set(spaces)
    return "".join((" " + ch) if i in pos else ch for i, ch in enumerate(s))

for (s, sp), want in examples:
    print(repr(add_spaces_set(s, sp)), "kỳ vọng", repr(want))

'Leetcode Helps Me Learn' kỳ vọng 'Leetcode Helps Me Learn'
'i code in py thon' kỳ vọng 'i code in py thon'
' s p a c i n g' kỳ vọng ' s p a c i n g'


**Độ phức tạp:** `O(n + m)` trung bình (tra `set` `O(1)`). Ưu điểm: **không cần** `spaces` sắp xếp — hữu ích khi đề biến thể cho vị trí lộn xộn.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 2: in `i`, ký tự, `j`, có chèn hay không, chuỗi đang dựng.

In [10]:
def trace_two_pointers(s, spaces):
    out, j = [], 0
    print(f"{'i':>2} | ch | j | spaces[j] | chèn? | kết quả tạm")
    for i, ch in enumerate(s):
        nxt = spaces[j] if j < len(spaces) else "-"
        ins = j < len(spaces) and spaces[j] == i
        if ins:
            out.append(" "); j += 1
        out.append(ch)
        print(f"{i:>2} | {ch}  | {j} | {str(nxt):>9} | {'có' if ins else '  '}    | {''.join(out)!r}")
    return "".join(out)

trace_two_pointers("icodeinpython", [1, 5, 7, 9])

 i | ch | j | spaces[j] | chèn? | kết quả tạm
 0 | i  | 0 |         1 |       | 'i'
 1 | c  | 1 |         1 | có    | 'i c'
 2 | o  | 1 |         5 |       | 'i co'
 3 | d  | 1 |         5 |       | 'i cod'
 4 | e  | 1 |         5 |       | 'i code'
 5 | i  | 2 |         5 | có    | 'i code i'
 6 | n  | 2 |         7 |       | 'i code in'
 7 | p  | 3 |         7 | có    | 'i code in p'
 8 | y  | 3 |         9 |       | 'i code in py'
 9 | t  | 4 |         9 | có    | 'i code in py t'
10 | h  | 4 |         - |       | 'i code in py th'
11 | o  | 4 |         - |       | 'i code in py tho'
12 | n  | 4 |         - |       | 'i code in py thon'


'i code in py thon'

In [11]:
def trace_slices(s, spaces):
    bounds = [0] + spaces + [len(s)]
    for a, b in zip(bounds, bounds[1:]):
        print(f"  mảnh s[{a}:{b}] = {s[a:b]!r}")
    print("  ->", repr(" ".join(s[a:b] for a, b in zip(bounds, bounds[1:]))))

trace_slices("spacing", [0, 1, 2, 3, 4, 5, 6])

  mảnh s[0:0] = ''
  mảnh s[0:1] = 's'
  mảnh s[1:2] = 'p'
  mảnh s[2:3] = 'a'
  mảnh s[3:4] = 'c'
  mảnh s[4:5] = 'i'
  mảnh s[5:6] = 'n'
  mảnh s[6:7] = 'g'
  -> ' s p a c i n g'


In [12]:
def trace_insert_right(s, spaces):
    print(f"bắt đầu: {s!r}")
    for p in reversed(spaces):
        s = s[:p] + " " + s[p:]
        print(f"  chèn trước chỉ số gốc {p}: {s!r}  (phần bên trái chỉ số {p} không bị dịch)")
    return s

trace_insert_right("LeetcodeHelpsMeLearn", [8, 13, 15])

bắt đầu: 'LeetcodeHelpsMeLearn'
  chèn trước chỉ số gốc 15: 'LeetcodeHelpsMe Learn'  (phần bên trái chỉ số 15 không bị dịch)
  chèn trước chỉ số gốc 13: 'LeetcodeHelps Me Learn'  (phần bên trái chỉ số 13 không bị dịch)
  chèn trước chỉ số gốc 8: 'Leetcode Helps Me Learn'  (phần bên trái chỉ số 8 không bị dịch)


'Leetcode Helps Me Learn'

## 9. Code nộp LeetCode

In [13]:
class Solution:
    def addSpaces(self, s: str, spaces: List[int]) -> str:
        out = []
        j, m = 0, len(spaces)
        for i, ch in enumerate(s):
            if j < m and spaces[j] == i:
                out.append(" ")
                j += 1
            out.append(ch)
        return "".join(out)

sol = Solution()
for (s, sp), want in examples:
    got = sol.addSpaces(s, sp)
    print(repr(got), "OK" if got == want else "SAI")

'Leetcode Helps Me Learn' OK
'i code in py thon' OK
' s p a c i n g' OK


## 10. Vì sao thuật toán đúng

Gọi `T` là chuỗi kết quả theo định nghĩa: `T = s` với một dấu cách đặt ngay trước `s[p]` cho mỗi `p ∈ spaces`.

**Invariant (Cách 2).** Sau khi xử lý `i` ký tự đầu, `out` bằng tiền tố của `T` gồm `s[0..i−1]` và các dấu cách trước chúng; `j = |{p ∈ spaces : p < i}|`.
- Ban đầu `i = 0`, `out` rỗng, `j = 0` ✓.
- Bước `i`: vì `spaces` tăng ngặt và mọi `p < i` đã được đếm, `spaces[j]` (nếu có) là vị trí chèn nhỏ nhất `≥ i`. Nếu nó bằng `i` thì `T` có đúng một dấu cách ngay trước `s[i]` ⇒ ghi `" "`, `j += 1`. Sau đó ghi `s[i]`. Invariant giữ.

Kết thúc `i = n`: `out = T` (mọi `p ≤ n − 1` đã xử lý). ✓

**Cách 1:** chèn ở `p` lớn nhất trước; với `p' < p`, ký tự `s[p']` vẫn ở chỉ số `p'` sau khi chèn ở `p` ⇒ mọi lần chèn đều đúng vị trí gốc. ✓

**Cách 3:** các mảnh `s[b_t:b_{t+1}]` ghép lại là `s`; `join` đặt đúng một dấu cách giữa hai mảnh liên tiếp, tức ngay trước `s[b_t]` với `t ≥ 1`. ✓

## 11. Phân tích Time / Space

`n = len(s) ≤ 3·10⁵`, `m = len(spaces) ≤ 3·10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Chèn từ phải | `O(m (n + m))` | `O(n + m)` |
| 2. Hai con trỏ + list | `O(n + m)` | `O(n + m)` |
| 3. Cắt lát + join | `O(n + m)` | `O(n + m)` |
| 4. Set + duyệt | `O(n + m)` trung bình | `O(n + m)` |

In [14]:
rng = random.Random(1)
n = 3 * 10**5
S = "".join(rng.choice(string.ascii_letters) for _ in range(n))
SP_all = list(range(n))                                  # m = n: chèn trước mọi ký tự
SP_some = sorted(rng.sample(range(n), 1000))
for name_, sp in [("m = 3e5", SP_all), ("m = 1000", SP_some)]:
    res = []
    for fname, fn in [("two_pointers", add_spaces_two_pointers), ("slices", add_spaces_slices), ("set", add_spaces_set), ("submit", Solution().addSpaces)]:
        t0 = time.perf_counter(); r = fn(S, sp); res.append(r)
        print(f"{name_:9s} {fname:12s}: {(time.perf_counter() - t0) * 1000:.1f} ms")
    assert len(set(res)) == 1 and len(res[0]) == n + len(sp)
t0 = time.perf_counter(); add_spaces_insert(S[:50000], list(range(0, 50000, 10))); print(f"chèn trực tiếp (n = 5e4, m = 5e3): {time.perf_counter() - t0:.2f}s")

m = 3e5   two_pointers: 22.9 ms
m = 3e5   slices      : 17.2 ms
m = 3e5   set         : 32.7 ms
m = 3e5   submit      : 20.2 ms


m = 1000  two_pointers: 12.6 ms
m = 1000  slices      : 0.2 ms
m = 1000  set         : 12.8 ms


m = 1000  submit      : 12.6 ms


chèn trực tiếp (n = 5e4, m = 5e3): 0.02s


## 12. Bộ test

In [15]:
fns = [add_spaces_two_pointers, add_spaces_slices, add_spaces_set, Solution().addSpaces]

for (s, sp), want in examples:
    assert add_spaces_insert(s, sp) == want
    for fn in fns:
        assert fn(s, sp) == want

edge = [(("a", [0]), " a"), (("ab", [1]), "a b"), (("ab", [0, 1]), " a b"), (("abc", [2]), "ab c"),
        (("Hello", [0]), " Hello"), (("xyz", [0, 2]), " xy z")]
for (s, sp), want in edge:
    assert add_spaces_insert(s, sp) == want, (s, sp)
    for fn in fns:
        assert fn(s, sp) == want, (fn, s, sp)

rng = random.Random(2024)
for _ in range(2000):
    n = rng.randint(1, 30); s = "".join(rng.choice("abcXYZ") for _ in range(n))
    sp = sorted(rng.sample(range(n), rng.randint(1, n)))
    want = add_spaces_insert(s, sp)
    assert want.replace(" ", "") == s and len(want) == n + len(sp)
    for fn in fns:
        assert fn(s, sp) == want, (fn, s, sp)
print("ví dụ + edge + 2000 bộ ngẫu nhiên khớp cách chèn trực tiếp")

ví dụ + edge + 2000 bộ ngẫu nhiên khớp cách chèn trực tiếp


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chèn từ trái sang phải mà không bù độ lệch

In [16]:
def wrong_left_to_right(s, spaces):
    for p in spaces:
        s = s[:p] + " " + s[p:]                                  # SAI: chỉ số gốc đã bị dịch
    return s

for (s, sp), want in examples:
    print(f"sai = {wrong_left_to_right(s, sp)!r} | đúng = {want!r}")

sai = 'Leetcode Help s MeLearn' | đúng = 'Leetcode Helps Me Learn'
sai = 'i cod e i npython' | đúng = 'i code in py thon'
sai = '       spacing' | đúng = ' s p a c i n g'


Sau `t` lần chèn, ký tự gốc ở chỉ số `p` nằm ở `p + t`. **Sửa:** chèn từ phải sang trái, hoặc dùng `p + t` (đoạn code ngay dưới), hoặc tốt nhất là không chèn trực tiếp.

In [17]:
def fixed_with_offset(s, spaces):
    for t, p in enumerate(spaces):
        s = s[:p + t] + " " + s[p + t:]                          # bù t dấu cách đã chèn
    return s

print([fixed_with_offset(s, sp) == want for (s, sp), want in examples])

[True, True, True]


### Lỗi 2 — Chèn **sau** ký tự thay vì trước

In [18]:
def wrong_after(s, spaces):
    out, j = [], 0
    for i, ch in enumerate(s):
        out.append(ch)
        if j < len(spaces) and spaces[j] == i:                   # SAI: dấu cách đặt sau s[i]
            out.append(" "); j += 1
    return "".join(out)

for (s, sp), want in examples:
    print(f"sai = {wrong_after(s, sp)!r} | đúng = {want!r}")

sai = 'LeetcodeH elpsM eL earn' | đúng = 'Leetcode Helps Me Learn'
sai = 'ic odei np yt hon' | đúng = 'i code in py thon'
sai = 's p a c i n g ' | đúng = ' s p a c i n g'


Đề: "inserted **before** the character at the given index". **Sửa:** ghi `" "` trước `s[i]`.

### Lỗi 3 — Ghép chuỗi `res += ch` trong vòng lặp (hiệu năng)

In [19]:
def concat_loop(s, spaces):
    res, j = "", 0
    for i, ch in enumerate(s):
        if j < len(spaces) and spaces[j] == i:
            res += " "; j += 1
        res += ch
    return res

t0 = time.perf_counter(); a = concat_loop(S, SP_all); t1 = time.perf_counter(); b = add_spaces_two_pointers(S, SP_all); t2 = time.perf_counter()
assert a == b
print(f"res += ch: {t1 - t0:.3f}s | list + join: {t2 - t1:.3f}s")

res += ch: 2.031s | list + join: 0.021s


Kết quả đúng nhưng chậm hơn hàng chục tới hàng trăm lần (xem số đo). CPython có tối ưu "nối tại chỗ" khi chuỗi chỉ có một tham chiếu, nhưng tối ưu này **không được bảo đảm** — nó phụ thuộc vào cách bộ cấp phát mở rộng vùng nhớ, vào phiên bản trình thông dịch (PyPy không có), và mất tác dụng khi còn tham chiếu khác tới `res`. Khi không áp dụng được, mỗi lần nối sao chép toàn bộ chuỗi ⇒ `O(n²)`. Thói quen an toàn: list + `join`.

### Lỗi 4 — Quên kiểm tra `j < m` (hết vị trí chèn)

In [20]:
def wrong_no_bound(s, spaces):
    out, j = [], 0
    try:
        for i, ch in enumerate(s):
            if spaces[j] == i:                                   # SAI: j có thể = len(spaces)
                out.append(" "); j += 1
            out.append(ch)
        return "".join(out)
    except IndexError as e:
        return f"<IndexError: {e}>"

for (s, sp), want in examples:
    print(f"kết quả = {wrong_no_bound(s, sp)!r} | đúng = {want!r}")

kết quả = '<IndexError: list index out of range>' | đúng = 'Leetcode Helps Me Learn'
kết quả = '<IndexError: list index out of range>' | đúng = 'i code in py thon'
kết quả = ' s p a c i n g' | đúng = ' s p a c i n g'


Sau khi dùng hết `spaces`, `spaces[j]` truy cập ngoài mảng (ví dụ 3 tình cờ không lỗi vì vị trí chèn cuối là ký tự cuối). Ở đây bắt lỗi để notebook chạy tiếp; trong code thật **phải** kiểm tra `j < len(spaces)`. **Sửa:** điều kiện kép như code nộp.

### Lỗi 5 — Dùng `split`/`join` theo ký tự hoa

Ví dụ 1 trông như "chèn trước chữ hoa" nhưng ví dụ 2 toàn chữ thường — đề dùng **chỉ số**, không dùng nội dung. Đoán quy luật từ ví dụ là lỗi tư duy phổ biến.

In [21]:
import re
def wrong_by_uppercase(s, spaces):
    return re.sub(r"(?<!^)(?=[A-Z])", " ", s)                    # SAI: không dùng spaces

for (s, sp), want in examples:
    print(f"sai = {wrong_by_uppercase(s, sp)!r} | đúng = {want!r}")

sai = 'Leetcode Helps Me Learn' | đúng = 'Leetcode Helps Me Learn'
sai = 'icodeinpython' | đúng = 'i code in py thon'
sai = 'spacing' | đúng = ' s p a c i n g'


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Dựng chuỗi kết quả từ nhiều mảnh ⇒ **list + `join`**, không chèn/ghép lặp.
- Hai dãy chỉ số tăng dần ⇒ **hai con trỏ**.
- Chèn/xóa nhiều vị trí trên dãy đang thay đổi ⇒ làm **từ phải sang trái** hoặc dựng mới.
- `" ".join(lát cắt)` là cách Pythonic khi các mốc đã sắp xếp.
- Họ bài: 1528 *Shuffle String*, 1592 *Rearrange Spaces Between Words*, 2810 *Faulty Keyboard*, 1844 *Replace All Digits with Characters*.

## 15. Biến thể của bài

1. **Chèn chuỗi bất kỳ** (không chỉ dấu cách) ở mỗi vị trí, có thể khác nhau: list các `(vị trí, chuỗi)`.
2. **`spaces` không sắp xếp / có trùng**: dùng `Counter` vị trí (chèn nhiều dấu cách trước cùng ký tự).
3. **Chèn sau mỗi `k` ký tự** (định dạng số tài khoản, ví dụ 1694 *Reformat Phone Number*).
4. **Thao tác ngược**: cho chuỗi đã chèn, khôi phục `s` và `spaces`.

In [22]:
from collections import Counter
def insert_many(s, ops):
    """Biến thể 1+2: ops = list (vị trí, chuỗi chèn), vị trí có thể trùng/không sắp xếp; giữ thứ tự ops cho cùng vị trí."""
    at = {}
    for p, t in ops:
        at.setdefault(p, []).append(t)
    out = []
    for i, ch in enumerate(s):
        out.extend(at.get(i, [])); out.append(ch)
    out.extend(at.get(len(s), []))
    return "".join(out)

def group_every_k(s, k, sep="-"):
    """Biến thể 3."""
    return sep.join(s[i:i + k] for i in range(0, len(s), k))

def recover(t):
    """Biến thể 4: từ chuỗi kết quả lấy lại (s, spaces)."""
    s_chars, spaces = [], []
    for ch in t:
        if ch == " ":
            spaces.append(len(s_chars))
        else:
            s_chars.append(ch)
    return "".join(s_chars), spaces

print(insert_many("abcd", [(2, "_"), (0, "<"), (4, ">"), (2, "|")]))
print(group_every_k("1234567890", 3))
for (s, sp), want in examples:
    assert recover(want) == (s, sp)
print("khôi phục đúng cho cả ba ví dụ")

<ab_|cd>
123-456-789-0
khôi phục đúng cho cả ba ví dụ


**Biến thể 5 — 1592 Rearrange Spaces Between Words:** cho chuỗi gồm các từ và dấu cách, phân phối lại dấu cách **đều** giữa các từ (phần dư đặt ở cuối). Cũng là bài "dựng chuỗi từ các mảnh" — dùng `join`.

In [23]:
def rearrange_spaces(text):
    words = text.split(); sp = text.count(" ")
    if len(words) == 1:
        return words[0] + " " * sp
    gap, rem = divmod(sp, len(words) - 1)
    return (" " * gap).join(words) + " " * rem

def rearrange_spaces_brute(text):
    words = text.split(); sp = text.count(" "); k = len(words) - 1
    best = None
    for g in range(sp + 1):                         # thử mọi độ rộng khoảng cách, chọn lớn nhất hợp lệ
        if k == 0 or g * k <= sp:
            best = g
    if k == 0: return words[0] + " " * sp
    return (" " * best).join(words) + " " * (sp - best * k)

rng = random.Random(16)
for _ in range(500):
    words = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 3))) for _ in range(rng.randint(1, 4))]
    text = ""
    for w in words: text += " " * rng.randint(0, 3) + w
    text += " " * rng.randint(0, 3)
    if text.count(" ") == 0 and len(words) > 1: continue
    assert rearrange_spaces(text) == rearrange_spaces_brute(text) and len(rearrange_spaces(text)) == len(text)
print(repr(rearrange_spaces("  this   is  a sentence ")), "|", repr(rearrange_spaces(" practice   makes   perfect")))

'this   is   a   sentence' | 'practice   makes   perfect '


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chèn từ phải | nghĩa đen | `O(m(n+m))` | `O(n+m)` | kiểm chứng, `m` nhỏ |
| 2. Hai con trỏ + list | dựng mới | `O(n+m)` | `O(n+m)` | **nộp bài** |
| 3. Cắt lát + join | mốc chia mảnh | `O(n+m)` | `O(n+m)` | nhanh nhất trong Python |
| 4. Set + duyệt | tra vị trí | `O(n+m)` | `O(n+m)` | `spaces` không sắp xếp |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `("abcdef", [3])`? `("abcdef", [0, 5])`? `("ab", [0, 1])`?

**Bài 2 (tính tay):** `s = "PythonIsFun"`, `spaces = [6, 8]`. Viết các mảnh của Cách 3 và kết quả.

**Bài 3 (điền TODO):** hoàn thiện `space_positions_in_result(spaces)` trả về chỉ số của các dấu cách **trong chuỗi kết quả**.

**Bài 4 (tự viết lại):** viết `add_spaces_unsorted(s, spaces)` nhận `spaces` **không sắp xếp** (vẫn không trùng) mà không dùng `set` — gợi ý: sắp xếp trước rồi dùng Cách 2.

**Bài 5 (mini challenge):** cho `s` và danh sách thao tác "chèn chuỗi `t` vào trước ký tự gốc ở chỉ số `p`" với **nhiều** thao tác có thể cùng `p` (giữ thứ tự thao tác), và thêm thao tác "xóa ký tự gốc ở chỉ số `q`". Dựng chuỗi kết quả trong `O(n + tổng độ dài)`. Kiểm chứng bằng mô phỏng trên list với `insert`/`del` xử lý từ phải sang trái. (Biến thể tự đặt — đối chiếu lại đề gốc nếu dùng làm đề.)

In [24]:
def space_positions_in_result(spaces):
    # TODO: dấu cách thứ t (0-based) nằm ở chỉ số spaces[t] + t
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Chèn trước chỉ số. (2) `"abc def"`, `" abcde f"`. (3) `" a b"`.

**Bài 2:** (1) Mốc `[0, 6, 8, 11]`. (2) Mảnh `"Python"`, `"Is"`, `"Fun"`. (3) `"Python Is Fun"`.

**Bài 3:** (1) Trước dấu cách thứ `t` đã có `t` dấu cách khác. (2) Vị trí = `spaces[t] + t`. (3) Kiểm tra bằng `[i for i, c in enumerate(res) if c == " "]`.

**Bài 4:** (1) `sorted(spaces)`. (2) Gọi Cách 2. (3) Độ phức tạp `O(n + m log m)`.

**Bài 5:** (1) `before[p]` = list chuỗi chèn trước `p`; `deleted` = set. (2) Duyệt `i`: ghi `before[i]`, rồi ghi `s[i]` nếu không bị xóa. (3) Mô phỏng chậm: xử lý thao tác theo chỉ số gốc giảm dần trên list ký tự.

## 19. Đáp án tham khảo

In [25]:
# Bài 1
assert [Solution().addSpaces(s, sp) for s, sp in [("abcdef", [3]), ("abcdef", [0, 5]), ("ab", [0, 1])]] == ["abc def", " abcde f", " a b"]
# Bài 2
trace_slices("PythonIsFun", [6, 8])
assert Solution().addSpaces("PythonIsFun", [6, 8]) == "Python Is Fun"

# Bài 3
def space_positions_in_result_ans(spaces):
    return [p + t for t, p in enumerate(spaces)]

# Bài 4
def add_spaces_unsorted_ans(s, spaces):
    return add_spaces_two_pointers(s, sorted(spaces))

rng = random.Random(41)
for _ in range(1000):
    n = rng.randint(1, 20); s = "".join(rng.choice("abc") for _ in range(n))
    sp = rng.sample(range(n), rng.randint(1, n))
    res = add_spaces_unsorted_ans(s, sp)
    assert res == add_spaces_set(s, sp)
    assert space_positions_in_result_ans(sorted(sp)) == [i for i, c in enumerate(res) if c == " "]
print("Bài 3 ví dụ 2:", space_positions_in_result_ans([1, 5, 7, 9]))
print("Bài 1-4: OK")

  mảnh s[0:6] = 'Python'
  mảnh s[6:8] = 'Is'
  mảnh s[8:11] = 'Fun'
  -> 'Python Is Fun'
Bài 3 ví dụ 2: [1, 6, 9, 12]
Bài 1-4: OK


In [26]:
# Bài 5 — chèn nhiều + xóa theo chỉ số gốc
def edit_string(s, inserts, deletes):
    before = {}
    for p, t in inserts:
        before.setdefault(p, []).append(t)
    dele = set(deletes); out = []
    for i, ch in enumerate(s):
        out.extend(before.get(i, ()))
        if i not in dele:
            out.append(ch)
    return "".join(out)

def edit_string_slow(s, inserts, deletes):
    chars = list(s)
    by_pos = {}
    for p, t in inserts: by_pos.setdefault(p, []).append(t)
    for p in sorted(set(by_pos) | set(deletes), reverse=True):   # từ phải sang trái: chỉ số gốc không dịch
        if p in set(deletes): del chars[p]
        for t in reversed(by_pos.get(p, [])):
            chars.insert(p, t)
    return "".join(chars)

rng = random.Random(55)
for _ in range(1000):
    n = rng.randint(1, 12); s = "".join(rng.choice("abcd") for _ in range(n))
    inserts = [(rng.randrange(n), rng.choice(["_", "--", "X", ""])) for _ in range(rng.randint(0, 5))]
    deletes = rng.sample(range(n), rng.randint(0, n))
    assert edit_string(s, inserts, deletes) == edit_string_slow(s, inserts, deletes)
print("Bài 5 ví dụ:", repr(edit_string("icodeinpython", [(1, " "), (5, " "), (7, " "), (9, " ")], [0])))

Bài 5 ví dụ: ' code in py thon'


## 20. Template nhớ nhanh + Checklist

```python
# Chèn dấu cách trước các chỉ số gốc (tăng dần)
out, j = [], 0
for i, ch in enumerate(s):
    if j < len(spaces) and spaces[j] == i:
        out.append(" "); j += 1
    out.append(ch)
return "".join(out)

# Một dòng: b = [0] + spaces + [len(s)]; " ".join(s[x:y] for x, y in zip(b, b[1:]))
```

**Checklist trước khi submit:**
- [ ] Chèn **trước** ký tự (không phải sau).
- [ ] Không chèn trực tiếp từ trái sang phải trên chuỗi đang đổi.
- [ ] Dùng list + `join`, không ghép chuỗi trong vòng lặp.
- [ ] Kiểm tra `j < len(spaces)`.
- [ ] Độ dài kết quả `= len(s) + len(spaces)`.